# Week 3 — Tree Ensembles & Boosting Showdown

**Theme:** Supervised learning II — decision trees, ensembles, and boosting

Last week we used k-NN and linear regression. This week we build up a whole
**family** of tree-based models, in the order they historically appeared and
build on each other:

1. **Decision Tree** — one tree of yes/no questions
2. **Random Forest** — many trees, trained independently, that vote together
   (*bagging*)
3. **AdaBoost** — many *weak* trees, trained one after another, each focusing
   on the previous one's mistakes (*boosting*)
4. **Gradient Boosting (GBM)** — boosting generalized: each new tree fits the
   *residual error* of the ensemble so far
5. **XGBoost** — a heavily optimized, regularized gradient boosting library
   (the long-time favorite for tabular data competitions)
6. **LightGBM** — an even faster gradient boosting library, using a different
   tree-growth strategy

All six train and get compared on the **same** dataset — a real, imbalanced
**open dataset from Kaggle** — so the comparison stays apples-to-apples. The
first three (Decision Tree, Random Forest, AdaBoost) use hand-picked
hyperparameters, explained as we go. Right after AdaBoost, we take a short
detour into **automated hyperparameter search** (GridSearchCV,
RandomizedSearchCV, HalvingRandomSearchCV) and use it to tune the remaining
three (Gradient Boosting, XGBoost, LightGBM), which have larger,
harder-to-hand-tune search spaces.

In [ ]:
!pip install -q xgboost lightgbm kagglehub

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.experimental import enable_halving_search_cv  # noqa: F401 -- required to unlock the next import
from sklearn.model_selection import HalvingRandomSearchCV
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

## Dataset: Calorie Burn Efficiency (Kaggle)

Every model in this notebook trains on the same real, open dataset —
**[Calorie Burn Efficiency](https://www.kaggle.com/datasets/parasharmanu/close-to-realistic-calorie-efficiency-dataset)**:
predicting a person's calorie-burn efficiency (Low / Moderate / High) from
13 activity and body metrics. It's a **3-class classification** problem, and
realistically messy — the classes are heavily imbalanced.

### 데이터 내려받기

`kagglehub`로 바로 내려받습니다 (공개 데이터셋이라 별도 로그인 없이 동작합니다).

In [ ]:
import os
import kagglehub

dataset_dir = kagglehub.dataset_download(
    "parasharmanu/close-to-realistic-calorie-efficiency-dataset"
)
csv_path = os.path.join(dataset_dir, "calorie_efficiency_dataset.csv")
print("Downloaded to:", csv_path)

In [ ]:
df_raw = pd.read_csv(csv_path)
print(df_raw.shape)
df_raw.head()

### 컬럼이 실제로 무엇을 의미하는지

데이터셋 제작자가 공개한 생성 방식에 따르면, 이 데이터는 다음 과정으로
만들어졌습니다:

1. **원시 비율**: `calories_burned / (steps_per_day + 20 × active_minutes)`
   에서 시작 (활동 시간이 걸음 수보다 20배 더 크게 반영됨 — 양보다 강도가
   중요하다는 의미)
2. 여기에 **가중치**를 더합니다: `muscle_mass_ratio`(+0.3, 가장 큰 긍정
   요인), `body_fat_percentage`(−0.2), `hydration_liters`(+0.05),
   `sleep_hours`(+0.05)
3. **심박수 보정**: `80 / heart_rate_resting` (낮을수록 좋음),
   `120 / heart_rate_avg` (너무 높으면 나쁨)
4. **연속 운동일수(`continuous_exercise_days`, 0~7) 보정**: 하루당 +3%,
   5일 이상이면 +10% 추가 보너스 — 단 6일 이상인데 `sleep_hours` < 6이면
   −10% 페널티 (회복 없는 무리한 연속 운동)
5. **하드 컷**: `workouts_per_week` > 6이면 −15% (과훈련), `sleep_hours` < 5면
   −25%
6. 최종적으로 0~10 사이로 정규화한 값이 바로 **`efficiency_score`**이고,
   `calorie_efficiency`는 이 점수를 **고정 구간**으로 나눈 것입니다
   (> 7 High / 4~7 Moderate / < 4 Low) — 단, 그중 **약 8%는 일부러 라벨을
   무작위로 섞어** 현실적인 노이즈를 흉내냈습니다.
7. `bmi`는 이 공식에 직접 쓰이진 않지만 참고용 특징으로 함께 제공됩니다.

즉 `efficiency_score`를 제외한 12개 특징은 전부 타깃을 만드는 **입력값**이라
정상적으로 학습해야 할 특징이지만, `efficiency_score` 자신은 타깃을 만드는
**마지막 단계의 값**이라 사실상 답에 가깝습니다 — 8%의 무작위 노이즈만 빼면요.

### `efficiency_score`가 실제로 얼마나 답에 가까운지 확인

클래스별 분포를 직접 보면, 뚜렷하게 갈리되 (~8% 정도의) 예외도 함께 보일
것입니다 — 바로 그 의도된 라벨 노이즈입니다.

In [ ]:
print(df_raw.groupby("calorie_efficiency")["efficiency_score"].describe())

df_raw.boxplot(column="efficiency_score", by="calorie_efficiency", figsize=(6, 4))
plt.title("efficiency_score by calorie_efficiency class")
plt.suptitle("")
plt.ylabel("efficiency_score")
plt.show()

거의 완벽하게 갈리는 것을 확인했으니, `efficiency_score`는 **정답을 거의
그대로 담고 있는 값(data leakage)**으로 보고 제거합니다 — 그래야 모델이
"점수를 그대로 구간으로 되돌리는" 트릭 대신, 12개의 진짜 행동/신체 지표에서
비선형 상호작용을 학습하게 됩니다 (이게 바로 이번 주차에서 tree 계열 모델을
쓰는 이유입니다).

클래스 분포를 보면 대부분 `Low Efficiency`인 매우 불균형한 데이터이기도
합니다. 이번 주차에서는 (7주차에서 배울 정밀한 불균형 처리 대신) 간단히
**클래스별로 같은 개수만큼 뽑아서** 균형 잡힌 부분집합으로 실습합니다.

In [ ]:
TARGET = "calorie_efficiency"

df = df_raw.drop(columns=["efficiency_score"])

print("Class distribution (raw):")
print(df[TARGET].value_counts())

# Feature engineering: a few interaction ratios (same idea as Week 1's pandas practice)
df["activity_intensity"] = df["active_minutes"] / (df["steps_per_day"] + 1)
df["fitness_ratio"] = df["muscle_mass_ratio"] / (df["body_fat_percentage"] + 1e-5)
df["recovery_score"] = df["sleep_hours"] * df["hydration_liters"]
df["cardio_efficiency"] = df["heart_rate_resting"] / (df["heart_rate_avg"] + 1)

# Balance the classes by downsampling every class to the size of the smallest one
# (capped at 500/class so training stays fast).
per_class = min(df[TARGET].value_counts().min(), 500)
df_balanced = pd.concat(
    [group.sample(per_class, random_state=42) for _, group in df.groupby(TARGET)],
    ignore_index=True,
)
print("\nClass distribution (balanced subset):")
print(df_balanced[TARGET].value_counts())

label_encoder = LabelEncoder()
X = df_balanced.drop(columns=[TARGET])
y = label_encoder.fit_transform(df_balanced[TARGET])
print("\nLabel mapping:", dict(zip(label_encoder.classes_, range(len(label_encoder.classes_)))))

# scikit-learn's own train_test_split, stratified so each split keeps the
# same class proportions -- a manual index/range cut can easily end up with
# a skewed train or test split, especially on a small, class-balanced dataset
# like this one.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"\nTrain: {len(X_train)} rows, Test: {len(X_test)} rows")

## 첫 세 모델: 손으로 고른 하이퍼파라미터로

Decision Tree, Random Forest, AdaBoost는 우선 하이퍼파라미터를 직접 하나씩
골라서 학습시킵니다. 매번 같은 작업(학습 → 시간 측정 → 예측 → 정확도 측정)을
반복하니 헬퍼 함수로 만들어 재사용합니다.

In [ ]:
results = []

def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    train_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    preds = model.predict(X_test)
    inference_time = time.perf_counter() - t0

    acc = accuracy_score(y_test, preds)
    results.append({
        "model": name,
        "accuracy": acc,
        "train_time_sec": train_time,
        "inference_time_sec": inference_time,
        "best_params": None,
    })
    print(f"[{name}] accuracy={acc:.3f}  train={train_time:.3f}s  inference={inference_time:.4f}s")
    return model

## 1. Decision Tree

**핵심 하이퍼파라미터 (모델을 만들 때 미리 정해줘야 하는 값들)**

- `criterion` — 어떤 기준으로 "가장 좋은 질문(분할)"을 고를지. 후보: `"gini"`,
  `"entropy"`, `"log_loss"`. **기본값: `"gini"`**
- `max_depth` — 트리가 내려갈 수 있는 최대 깊이. 얕을수록 단순(과소적합 위험),
  깊을수록 복잡(과대적합 위험). 후보: 양의 정수 또는 `None`(제한 없음).
  **기본값: `None`**
- `min_samples_split` — 한 노드를 더 나누기 위해 필요한 최소 샘플 수. 후보:
  정수(개수) 또는 0~1 사이 실수(비율). **기본값: `2`**
- `min_samples_leaf` — 리프(끝마디)에 남아야 하는 최소 샘플 수. 값이 클수록
  더 단순한(덜 과대적합된) 트리가 됨. **기본값: `1`**
- `max_features` — 분할을 고를 때 고려할 특성(feature) 개수. 후보: `"sqrt"`,
  `"log2"`, 정수/실수, 또는 `None`(전체 사용). **기본값: `None`**
- `class_weight` — 클래스 불균형을 보정할지. 후보: `None`, `"balanced"`, 또는
  직접 지정한 딕셔너리. **기본값: `None`**

여기서는 트리를 시각화하기 쉽도록 `max_depth=3`으로 일부러 얕게 만듭니다
(자동으로 하이퍼파라미터를 탐색하는 도구는 AdaBoost 다음에 배웁니다).

In [ ]:
tree = evaluate_model(
    "Decision Tree",
    DecisionTreeClassifier(max_depth=3, random_state=42),
    X_train, y_train, X_test, y_test,
)

plt.figure(figsize=(14, 7))
plot_tree(tree, feature_names=X.columns, class_names=label_encoder.classes_,
          filled=True, fontsize=8)
plt.title("Decision Tree (max_depth=3)")
plt.show()

## 2. Random Forest (bagging)

한 그루의 트리는 데이터가 조금만 바뀌어도 결과가 크게 흔들릴 수 있습니다 (분산이
큼). **랜덤 포레스트**는 데이터와 특성을 각각 무작위로 조금씩 다르게 뽑아 **여러
그루의 트리를 독립적으로** 학습시킨 뒤, 다수결(분류) 또는 평균(회귀)으로 최종
예측을 냅니다 — 이 방식을 **배깅(bagging, bootstrap aggregating)**이라 부릅니다.

**핵심 하이퍼파라미터**

- `n_estimators` — 트리를 몇 그루 만들지. 많을수록 보통 더 안정적이지만 느려짐.
  후보: 양의 정수. **기본값: `100`**
- `max_depth` — 트리 하나하나의 최대 깊이 (Decision Tree와 동일 의미).
  **기본값: `None`**
- `max_features` — 각 분할에서 고려할 특성 개수 (트리마다 무작위로 다르게 뽑는
  핵심 아이디어). 후보: `"sqrt"`, `"log2"`, 정수/실수, `None`.
  **기본값: `"sqrt"`**
- `min_samples_leaf` — Decision Tree와 동일 의미. **기본값: `1`**
- `bootstrap` — 각 트리를 학습시킬 때 데이터를 복원추출(bootstrap)할지. 후보:
  `True`/`False`. **기본값: `True`**
- `class_weight` — Decision Tree와 동일. **기본값: `None`**

In [ ]:
forest = evaluate_model(
    "Random Forest",
    RandomForestClassifier(n_estimators=100, random_state=42),
    X_train, y_train, X_test, y_test,
)

importances = pd.Series(forest.feature_importances_, index=X.columns)
importances.sort_values(ascending=False).head(10).plot(kind="barh", figsize=(6, 4))
plt.title("Top 10 Most Important Features (Random Forest)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Part 2 — Boosting

Decision Tree와 Random Forest는 트리들을 **독립적으로** 학습시켰습니다
(bagging). 지금부터 배울 **AdaBoost / Gradient Boosting / XGBoost /
LightGBM**은 전부 **부스팅(boosting)** 계열입니다: 트리를 하나씩 **순서대로**
학습시키면서, 매번 "이전까지의 앙상블이 틀린 부분"에 집중합니다. 데이터셋은
계속 같은 Calorie Efficiency 데이터를 씁니다.

## 3. AdaBoost (Adaptive Boosting)

첫 번째 부스팅 알고리즘입니다. 얕은 트리(보통 깊이 1의 "stump")를 하나씩
학습시키되, **직전 트리가 틀린 샘플의 가중치를 높여서** 다음 트리가 그 부분에
더 집중하게 만듭니다. 최종 예측은 각 트리의 (성능에 따라 가중된) 투표입니다.

**핵심 하이퍼파라미터**

- `estimator` — 부스팅할 약한 학습기(weak learner). 후보: 어떤 분류기든 가능,
  보통 얕은 트리. **기본값: `None`** → 내부적으로 깊이 1짜리
  `DecisionTreeClassifier` (decision stump) 사용
- `n_estimators` — 순차적으로 몇 개의 약한 학습기를 쌓을지. 후보: 양의 정수.
  **기본값: `50`**
- `learning_rate` — 각 학습기의 기여도를 얼마나 줄여서 반영할지 (작을수록
  보수적, `n_estimators`를 늘려 보완해야 함). 후보: 양의 실수. **기본값: `1.0`**

In [ ]:
ada = evaluate_model(
    "AdaBoost",
    AdaBoostClassifier(n_estimators=100, learning_rate=1.0, random_state=42),
    X_train, y_train, X_test, y_test,
)

## 하이퍼파라미터 자동 탐색 (Hyperparameter Search)

지금까지처럼 하이퍼파라미터를 손으로 하나씩 골라 넣을 수도 있지만, 남은 세
모델(Gradient Boosting, XGBoost, LightGBM)은 조정할 하이퍼파라미터가 훨씬
많아서 손으로 최적값을 찾기 어렵습니다. 실전에서는 여러 후보 조합을
**자동으로** 시도해보고 검증 성능이 가장 좋은 조합을 골라주는 도구를 씁니다.
scikit-learn에 내장된 세 가지를 소개합니다.

### 1. GridSearchCV — 모든 조합을 다 시도

지정한 하이퍼파라미터 후보들의 **모든 조합**을 교차검증(cross-validation)으로
평가해서 가장 좋은 조합을 찾습니다. 조합 수가 많아지면 학습 횟수가
기하급수적으로 늘어납니다 (예: 4×3×2 = 24가지 조합 × `cv=3` = 72번 학습).

**핵심 파라미터**

- `estimator` — 튜닝할 모델 (필수)
- `param_grid` — 탐색할 하이퍼파라미터 후보. 후보: `{"파라미터명": [값들, ...]}`
  형태의 dict, 또는 이런 dict들의 list (필수)
- `scoring` — 어떤 지표로 "좋다"를 판단할지. 후보: `"accuracy"`,
  `"f1_macro"`, `"roc_auc"` 등 문자열 또는 커스텀 scorer. **기본값: `None`**
  → 모델의 기본 `.score()` 사용
- `cv` — 교차검증 폴드 수(정수) 또는 splitter 객체. **기본값: `None`** →
  5-fold (분류 문제는 자동으로 `StratifiedKFold` 사용 — 클래스 비율이
  유지된 채로 나뉘므로, 직접 범위를 잘라 나누는 것보다 안전합니다)
- `n_jobs` — 병렬로 쓸 CPU 코어 수. 후보: 정수 또는 `-1`(전부 사용).
  **기본값: `None`** (1개)
- `refit` — 탐색이 끝난 뒤 최적 조합으로 전체 학습 데이터에 다시 학습할지.
  **기본값: `True`**

작은 예시로 Decision Tree에 적용해봅니다 (이미 위에서 `max_depth=3`으로
직접 골랐던 것과 비교해보세요).

In [ ]:
grid_search_demo = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid={
        "max_depth": [2, 3, 4, 5],
        "min_samples_leaf": [1, 5, 10],
    },
    scoring="accuracy",
    cv=3,
)
grid_search_demo.fit(X_train, y_train)
print("Best params:", grid_search_demo.best_params_)
print("Best CV accuracy:", f"{grid_search_demo.best_score_:.3f}")

### 2. RandomizedSearchCV — 무작위로 일부만 시도

모든 조합을 다 보는 대신, 지정한 횟수(`n_iter`)만큼 **무작위로** 조합을 뽑아
평가합니다. 후보 조합이 아주 많을 때 GridSearchCV보다 훨씬 빠르면서도 비슷한
성능을 찾아내는 경우가 많습니다.

**핵심 파라미터**

- `estimator`, `param_distributions` — GridSearchCV의 `param_grid`와 같은
  역할이지만, 값 리스트뿐 아니라 `scipy.stats`의 확률분포도 지정할 수
  있습니다 (연속값 탐색에 유리). (필수)
- `n_iter` — 무작위로 몇 개의 조합을 시도할지. 후보: 양의 정수.
  **기본값: `10`**
- `scoring`, `cv`, `n_jobs` — GridSearchCV와 동일한 의미/기본값
- `random_state` — 무작위 샘플링을 재현하기 위한 시드. **기본값: `None`**

Random Forest에 적용해봅니다.

In [ ]:
random_search_demo = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions={
        "n_estimators": [50, 100, 200, 300],
        "max_depth": [4, 6, 8, None],
        "max_features": ["sqrt", "log2"],
    },
    n_iter=8,
    scoring="accuracy",
    cv=3,
    random_state=42,
)
random_search_demo.fit(X_train, y_train)
print("Best params:", random_search_demo.best_params_)
print("Best CV accuracy:", f"{random_search_demo.best_score_:.3f}")

### 3. HalvingRandomSearchCV — 단계적으로 후보를 좁혀가기 (successive halving)

*(실험적 기능이라 `from sklearn.experimental import enable_halving_search_cv`를
먼저 import해야 활성화됩니다 — 이미 위에서 했습니다.)*

처음 라운드에는 적은 데이터로 **많은** 후보 조합을 빠르게 걸러내고, 라운드가
진행될수록 살아남은 유망한 조합만 **점점 더 많은** 데이터로 재평가합니다.
가망 없는 조합에 시간을 낭비하지 않아서, 후보가 아주 많을 때 특히 효율적입니다.

**핵심 파라미터**

- `estimator`, `param_distributions` — 위와 동일
- `n_candidates` — 첫 라운드에서 시도할 후보 조합 개수. 후보: 정수 또는
  `"exhaust"`(자원 예산에 맞춰 자동 결정). **기본값: `"exhaust"`**
- `factor` — 매 라운드마다 후보를 몇 분의 1로 줄일지. 후보: 1보다 큰 수.
  **기본값: `3`**
- `resource` — 라운드마다 늘려갈 "자원"이 무엇인지. 후보: `"n_samples"`
  (데이터 양) 또는 모델의 정수형 파라미터 이름(예: `"n_estimators"`).
  **기본값: `"n_samples"`**
- `min_resources` — 첫 라운드에서 쓸 최소 자원량. 후보: `"smallest"`,
  `"exhaust"`, 또는 정수. **기본값: `"smallest"`**
- `cv`, `scoring`, `random_state` — 위와 동일 (단 `cv` **기본값은 `5`**로,
  다른 두 방법과 달리 `None`이 아닙니다)

XGBoost에 적용해봅니다 — 단 `resource`를 기본값 `"n_samples"`(데이터 양) 대신
**`"n_estimators"`(트리 개수)**로 지정합니다. 이 데이터셋은 이미 500개/클래스로
작게 줄여둔 데다 클래스가 3개라, "데이터 양"을 자원으로 줄이면 초반 라운드의
작은 표본에 특정 클래스가 하나도 안 들어가서 학습이 실패할 수 있습니다 —
"트리 개수"를 자원으로 쓰면 이 문제 없이, 트리 개수가 많아질수록 느려지는
XGBoost의 특성을 그대로 halving으로 체감할 수 있습니다. (`resource`를
`"n_samples"`가 아닌 값으로 바꾸면 그 파라미터는 `param_distributions`가 아니라
`min_resources`/`max_resources`로 직접 범위를 지정합니다.)

In [ ]:
halving_search_demo = HalvingRandomSearchCV(
    XGBClassifier(eval_metric="mlogloss", random_state=42, n_jobs=1),
    param_distributions={
        "max_depth": [3, 4, 6, 8],
        "learning_rate": [0.01, 0.05, 0.1, 0.3],
    },
    resource="n_estimators",
    min_resources=20,
    max_resources=300,
    factor=3,
    cv=3,
    scoring="accuracy",
    random_state=42,
)
halving_search_demo.fit(X_train, y_train)
print("Best params:", halving_search_demo.best_params_)
print("Best CV accuracy:", f"{halving_search_demo.best_score_:.3f}")

### 이제부터: 남은 3개 모델엔 RandomizedSearchCV를 사용

셋 다 써봤으니, Gradient Boosting / XGBoost / LightGBM부터는 속도와 탐색 폭의
균형이 좋은 **RandomizedSearchCV**를 적용합니다. `evaluate_model`과 같은
패턴이지만, 학습 전에 탐색이 한 번 더 들어가는 버전입니다.

In [ ]:
def tune_and_evaluate(name, model, param_distributions, n_iter=15, cv=3):
    search = RandomizedSearchCV(
        model,
        param_distributions=param_distributions,
        n_iter=n_iter,
        scoring="accuracy",
        cv=cv,
        random_state=42,
        n_jobs=-1,
    )

    t0 = time.perf_counter()
    search.fit(X_train, y_train)
    train_time = time.perf_counter() - t0

    best_model = search.best_estimator_
    t0 = time.perf_counter()
    preds = best_model.predict(X_test)
    inference_time = time.perf_counter() - t0

    acc = accuracy_score(y_test, preds)
    results.append({
        "model": name,
        "accuracy": acc,
        "train_time_sec": train_time,
        "inference_time_sec": inference_time,
        "best_params": search.best_params_,
    })
    print(f"[{name}] best_params={search.best_params_}")
    print(f"[{name}] accuracy={acc:.3f}  search+train={train_time:.2f}s  inference={inference_time:.4f}s")
    return best_model

## 4. Gradient Boosting (GBM)

AdaBoost를 일반화한 아이디어입니다. 매 단계마다 "정답 - 지금까지의 예측"이라는
**잔차(residual)**를 계산하고, 다음 트리는 그 잔차 자체를 예측하도록
학습시킵니다 — 경사하강법(gradient descent)으로 손실을 줄여나가는 것과 같은
원리라 "gradient" boosting이라 부릅니다.

**핵심 하이퍼파라미터**

- `loss` — 최적화할 손실 함수. 후보: `"log_loss"`, `"exponential"`.
  **기본값: `"log_loss"`**
- `learning_rate` — 각 트리의 기여도를 줄이는 축소 계수(shrinkage). 후보: 양의
  실수. **기본값: `0.1`**
- `n_estimators` — 순차적으로 쌓을 트리 개수. 후보: 양의 정수. **기본값: `100`**
- `subsample` — 각 트리를 학습할 때 사용할 데이터 비율 (1.0 미만이면 일종의
  확률적 경사하강 + 정규화 효과). 후보: 0~1 사이 실수. **기본값: `1.0`**
- `max_depth` — 트리 하나하나의 최대 깊이. 후보: 양의 정수. **기본값: `3`**

In [ ]:
gbm_param_dist = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "max_depth": [2, 3, 4],
    "subsample": [0.7, 0.85, 1.0],
}

gbm = tune_and_evaluate("Gradient Boosting", GradientBoostingClassifier(random_state=42), gbm_param_dist)

## 5. XGBoost (eXtreme Gradient Boosting)

Gradient Boosting을 실전용으로 크게 최적화/정규화한 전문 라이브러리입니다.
결측치를 알아서 처리하고, 병렬로 트리를 빠르게 만들고, 과대적합을 막는
정규화 항(`reg_alpha`, `reg_lambda`)을 손실 함수에 직접 포함시킵니다. 오랫동안
정형(tabular) 데이터 경진대회의 표준 도구였습니다.

**핵심 하이퍼파라미터**

- `n_estimators` — 트리 개수. 후보: 양의 정수. **기본값: `100`**
- `learning_rate` (`eta`) — 축소 계수. 후보: 0~1 사이 실수. **기본값: `0.3`**
- `max_depth` — 트리 최대 깊이. 후보: 양의 정수. **기본값: `6`**
- `subsample` — 트리마다 사용할 데이터 비율. 후보: 0~1 사이 실수.
  **기본값: `1`**
- `colsample_bytree` — 트리마다 사용할 특성 비율. 후보: 0~1 사이 실수.
  **기본값: `1`**
- `reg_lambda` — L2 정규화 강도 (클수록 더 단순한 모델). 후보: 0 이상 실수.
  **기본값: `1`**
- `reg_alpha` — L1 정규화 강도. 후보: 0 이상 실수. **기본값: `0`**

In [ ]:
xgb_param_dist = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.05, 0.1, 0.3],
    "max_depth": [3, 4, 6, 8],
    "subsample": [0.7, 0.85, 1.0],
    "colsample_bytree": [0.7, 0.85, 1.0],
}

xgb = tune_and_evaluate(
    "XGBoost",
    XGBClassifier(eval_metric="mlogloss", random_state=42, n_jobs=1),
    xgb_param_dist,
)

## 6. LightGBM

Microsoft가 만든 또 다른 gradient boosting 라이브러리입니다. XGBoost가 트리를
레벨(깊이) 단위로 균형 있게 키우는 반면, LightGBM은 **손실을 가장 많이
줄이는 리프(leaf)를 골라서 그 방향으로만 먼저 키우는** 전략(leaf-wise
growth)을 써서 보통 더 빠르고, 대용량 데이터에 특히 강합니다.

**핵심 하이퍼파라미터**

- `n_estimators` — 트리 개수. 후보: 양의 정수. **기본값: `100`**
- `learning_rate` — 축소 계수. 후보: 0~1 사이 실수. **기본값: `0.1`**
- `num_leaves` — 트리 하나당 최대 리프 개수 (leaf-wise 성장의 핵심 파라미터 —
  깊이 대신 이것으로 복잡도를 제어). 후보: 양의 정수. **기본값: `31`**
- `max_depth` — 트리 최대 깊이 (`-1`은 제한 없음, `num_leaves`가 사실상 더
  중요). 후보: 양의 정수 또는 `-1`. **기본값: `-1`**
- `min_child_samples` — 리프 하나에 필요한 최소 샘플 수 (과대적합 방지).
  후보: 양의 정수. **기본값: `20`**

In [ ]:
lgbm_param_dist = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.05, 0.1, 0.3],
    "num_leaves": [15, 31, 63],
    "min_child_samples": [5, 10, 20, 30],
}

lgbm = tune_and_evaluate(
    "LightGBM",
    LGBMClassifier(random_state=42, verbose=-1, n_jobs=1),
    lgbm_param_dist,
)

## 전체 요약: 6개 모델 한눈에 비교

모두 같은 데이터셋과 같은 train/test split으로 비교했지만, 튜닝 방법은
다릅니다 — Decision Tree/Random Forest/AdaBoost는 손으로 고른 값, Gradient
Boosting/XGBoost/LightGBM은 RandomizedSearchCV로 자동 탐색한 값입니다
(`best_params`가 `None`이면 손으로 고른 모델이라는 뜻).

In [ ]:
summary = pd.DataFrame(results)
display(summary[["model", "accuracy", "train_time_sec", "inference_time_sec"]])
print("\n각 모델의 하이퍼파라미터 (None이면 손으로 고른 값을 그대로 사용):")
for row in results:
    print(f"- {row['model']}: {row['best_params']}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(summary["model"], summary["accuracy"], color="steelblue")
axes[0].set_title("Accuracy by Model")
axes[0].set_ylabel("Accuracy")
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(summary["model"], summary["inference_time_sec"], color="indianred")
axes[1].set_title("Inference Time by Model")
axes[1].set_ylabel("Seconds (test set)")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## Try it yourself

1. **Widen a search space.** Pick one model and add a hyperparameter (or
   more candidate values) to its `param_distributions` — does `best_score_`
   improve? Does search time?
2. **Tune the first three too.** Wrap Decision Tree, Random Forest, or
   AdaBoost with `tune_and_evaluate` instead of `evaluate_model` and compare
   — how much does search improve on the hand-picked values?
3. **Change the balanced sample size.** Try `per_class = 200` vs `800` —
   does accuracy change much? Does training time?
4. **Compare libraries, not just accuracy.** For a model you'd have to
   retrain daily on fresh data, would you pick the most accurate model, or
   the fastest one? What if you had 10 million rows instead of 1,500?

---
## 🎯 캡스톤: XGBoost로 주식 종가 예측하기

지금까지 배운 6개 모델 중 가장 강력했던 **XGBoost**를 실전 회귀(regression)
문제에 적용해봅니다: 과거 주가 데이터로 **다음 날 종가**를 예측하는 모델을
만들어보세요.

아래 데이터 수집/피처 엔지니어링 코드는 그대로 실행하면 되고, 여러분은
**모델을 학습시키고, 예측하고, 평가하는 부분**을 직접 작성합니다.

In [ ]:
# 주가 데이터 내려받기 + 피처 엔지니어링 (실행만 하면 됩니다)
!pip install -q yfinance

import yfinance as yf

TICKER = "AAPL"  # 원하는 종목으로 바꿔도 됩니다 (예: "005930.KS" = 삼성전자)

# auto_adjust=False로 받아야 "종가(Close)"와 "수정 종가(Adj Close)"가 별도
# 컬럼으로 남습니다 (True면 배당/액면분할이 반영된 조정가로 Close 자체를
# 덮어써서 Adj Close가 사라집니다).
price_df = yf.download(TICKER, period="3y", interval="1d", progress=False, auto_adjust=False)
price_df = price_df[["Open", "High", "Low", "Close", "Adj Close", "Volume"]].dropna()
price_df.columns = ["open", "high", "low", "close", "adj_close", "volume"]

# 이동평균선: 5일선/20일선/60일선/120일선 (전부 종가 기준)
price_df["ma5"] = price_df["close"].rolling(5).mean()
price_df["ma20"] = price_df["close"].rolling(20).mean()
price_df["ma60"] = price_df["close"].rolling(60).mean()
price_df["ma120"] = price_df["close"].rolling(120).mean()

# 예측 타깃: "다음 날" 종가
price_df["target_next_close"] = price_df["close"].shift(-1)
price_df = price_df.dropna()

feature_cols = ["open", "high", "low", "close", "adj_close", "volume", "ma5", "ma20", "ma60", "ma120"]
X_stock = price_df[feature_cols]
y_stock = price_df["target_next_close"]

# 시계열이므로 scikit-learn의 랜덤 train_test_split을 쓰지 않습니다 -- 그러면
# "미래" 데이터가 학습셋에 섞여 들어가 시험 점수가 실제보다 부풀려집니다
# (시계열의 shuffle=True는 미래 정보 누출입니다). 대신 "과거로 학습, 최근으로
# 평가"하도록 시간 순서 그대로 앞 80%/뒤 20%로 나눕니다.
split_idx = int(len(price_df) * 0.8)
X_stock_train, X_stock_test = X_stock.iloc[:split_idx], X_stock.iloc[split_idx:]
y_stock_train, y_stock_test = y_stock.iloc[:split_idx], y_stock.iloc[split_idx:]
dates_test = price_df.index[split_idx:]

print(f"Train: {len(X_stock_train)} rows, Test: {len(X_stock_test)} rows")
price_df.tail()

### 여러분의 과제

1. `XGBRegressor`를 만들고 `X_stock_train`, `y_stock_train`으로 학습시키세요.
   (힌트: 회귀 문제이므로 `XGBClassifier`가 아니라 `XGBRegressor`를 씁니다.)
2. 학습된 모델로 `X_stock_test`에 대한 종가를 예측하세요.
3. **실제 종가 vs 예측 종가**를 같은 그래프에 그리고, RMSE를 계산해 출력하세요.
   (힌트: `from sklearn.metrics import mean_squared_error`, 이후
   `mean_squared_error(y_true, y_pred) ** 0.5`)
4. **(자유 과제)** `TICKER`를 다른 종목으로 바꾸거나, `target_next_close`를
   "다음 날" 대신 "5일 뒤" 종가로 바꿔서 실험해보세요. 예측이 더 어려워지나요?

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

# TODO 1: XGBRegressor를 만들고 학습시키세요.


# TODO 2: X_stock_test에 대해 예측하세요.


# TODO 3: 실제 vs 예측 종가를 그래프로 그리고, RMSE를 계산해 출력하세요.
